# YOLOP Paper Reproduction Notebook

This notebook reproduces the YOLOP training and evaluation workflow on BDD100K using the same multi-task setup as the paper:
- object detection
- drivable area segmentation
- lane line segmentation

The main path uses the official `hustvl/YOLOP` repository and its train/test/demo scripts. The local `yolop_model.py` file is kept as a sanity-check fallback, but the reproduction flow below follows the paper setup.

## 1. Load Paper Reproduction Config

This section defines the paper-specific settings in one place: dataset paths, split names, model hyperparameters, random seeds, and evaluation targets.

In [1]:
from pathlib import Path
import os
import random
import json
import subprocess
import sys
import textwrap

ROOT = Path.cwd()
DATA_ROOT = ROOT / "datasets"
YOLOP_REPO = ROOT / "YOLOP"
RUN_NAME = "yolop-paper-reproduction"
SEED = 42
IMG_SIZE = 640
EPOCHS = 300
BATCH_SIZE = 16
DEVICE = "cuda" if os.environ.get("CUDA_VISIBLE_DEVICES", "") != "" else "cpu"

# Your actual dataset structure
EXPECTED_SPLITS = {
    "bdd100k_images_100k/100k/train",
    "bdd100k_images_100k/100k/val",
    "bdd100k_images_100k/100k/test",
    "bdd100k_drivable_maps/color_labels/train",
    "bdd100k_drivable_maps/color_labels/val",
    "bdd100k_drivable_maps/labels/train",
    "bdd100k_drivable_maps/labels/val",
    "bdd100k_labels/100k/train",
    "bdd100k_labels/100k/val",
    "bdd100k_labels/100k/test",
}

REPORT_METRICS = {
    "det": "mAP@0.5",
    "da_seg": "drivable-area IoU",
    "ll_seg": "lane-line IoU",
}

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print({
    "ROOT": str(ROOT),
    "DATA_ROOT": str(DATA_ROOT),
    "YOLOP_REPO": str(YOLOP_REPO),
    "RUN_NAME": RUN_NAME,
    "SEED": SEED,
    "IMG_SIZE": IMG_SIZE,
    "EPOCHS": EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "DEVICE": DEVICE,
})

{'ROOT': 'c:\\Users\\joaco\\OneDrive\\Documents\\facultad\\Vision\\TP-final-Vision-Artificial', 'DATA_ROOT': 'c:\\Users\\joaco\\OneDrive\\Documents\\facultad\\Vision\\TP-final-Vision-Artificial\\datasets', 'YOLOP_REPO': 'c:\\Users\\joaco\\OneDrive\\Documents\\facultad\\Vision\\TP-final-Vision-Artificial\\YOLOP', 'RUN_NAME': 'yolop-paper-reproduction', 'SEED': 42, 'IMG_SIZE': 640, 'EPOCHS': 300, 'BATCH_SIZE': 16, 'DEVICE': 'cpu'}


## 2. Set Up Environment and Dependencies

This section verifies the runtime, reports package versions, and initializes deterministic settings where possible.

In [2]:
import platform
import importlib.util
from pprint import pprint

required_packages = ["numpy", "torch", "cv2", "yaml", "matplotlib"]
package_status = {}
for name in required_packages:
    package_status[name] = importlib.util.find_spec(name) is not None

print("Python:", sys.version)
print("Platform:", platform.platform())
pprint(package_status)

# Make the run as reproducible as possible.
try:
    import numpy as np
    np.random.seed(SEED)
except Exception as exc:
    print("NumPy not ready:", exc)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
except Exception as exc:
    print("Torch not ready:", exc)

Python: 3.10.1 (tags/v3.10.1:2cd268a, Dec  6 2021, 19:10:37) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
{'cv2': True, 'matplotlib': True, 'numpy': True, 'torch': False, 'yaml': True}
Torch not ready: No module named 'torch'


## 3. Load Dataset and Paper Train/Test Split

YOLOP was trained and evaluated on BDD100K using the official image split plus three annotation streams: detection, drivable area segmentation, and lane-line segmentation.

This notebook expects the same paper-style folder layout.

### Download the BDD100K dataset with KaggleHub

This notebook uses the Kaggle mirror as a practical way to download the BDD100K assets. The paper split should still be preserved by using the train/val files already provided by the dataset rather than making a new random split.

In [3]:
# If needed, install kagglehub first.
# !pip install -q kagglehub

import shutil
from pathlib import Path
# DATA_ROOT = Path('/path/to/your/BDD100K')


In [4]:
from pathlib import Path

# Mirror the paper's required structure.
missing = [str((DATA_ROOT / rel)) for rel in EXPECTED_SPLITS if not (DATA_ROOT / rel).exists()]
print("Expected paper folders checked against:")
for rel in sorted(EXPECTED_SPLITS):
    print(" -", DATA_ROOT / rel)

if missing:
    print("\nMissing paper-format folders:")
    for item in missing:
        print(" -", item)
    print("\nThe official YOLOP training code requires the additional annotation folders from BDD100K.")
else:
    print("All expected paper-format folders are present.")

# Inspect what the current workspace contains.
for child in sorted(DATA_ROOT.iterdir()) if DATA_ROOT.exists() else []:
    print(child.name, "folder" if child.is_dir() else "file")

Expected paper folders checked against:
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_drivable_maps\color_labels\train
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_drivable_maps\color_labels\val
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_drivable_maps\labels\train
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_drivable_maps\labels\val
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_images_100k\100k\test
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_images_100k\100k\train
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_images_100k\100k\val
 - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_la

## 4. Apply Paper-Specific Preprocessing

YOLOP uses the paper's image resize and letterbox pipeline and expects the official BDD100K annotations prepared for detection, drivable area segmentation, and lane-line segmentation.

If your dataset is still in the raw BDD100K form, you need to generate the paper-format labels before training.

In [5]:
import subprocess
import sys
import shutil


def run(cmd, cwd=None):
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if not YOLOP_REPO.exists():
    run(["git", "clone", "https://github.com/hustvl/YOLOP.git", str(YOLOP_REPO)])
else:
    print("YOLOP repository already present:", YOLOP_REPO)

requirements = YOLOP_REPO / "requirements.txt"
if requirements.exists():
    run([sys.executable, "-m", "pip", "install", "-r", str(requirements)])
else:
    print("requirements.txt not found in cloned repository.")

# The paper's preprocessing is mostly about preparing BDD100K into YOLOP's expected folder layout.
preprocess_notes = textwrap.dedent(
    f"""
    Your dataset structure:
      - {DATA_ROOT / 'bdd100k_images_100k/100k/train'} (training images)
      - {DATA_ROOT / 'bdd100k_images_100k/100k/val'} (validation images)
      - {DATA_ROOT / 'bdd100k_images_100k/100k/test'} (test images)
      - {DATA_ROOT / 'bdd100k_labels'} (detection labels)
      - {DATA_ROOT / 'bdd100k_drivable_maps'} (segmentation maps)

    These will be used for evaluation. Ensure all directories are properly populated.
    """
)
print(preprocess_notes)


YOLOP repository already present: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\YOLOP

$ c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\venv\Scripts\python.exe -m pip install -r c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\YOLOP\requirements.txt

Your dataset structure:
  - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_images_100k\100k\train (training images)
  - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_images_100k\100k\val (validation images)
  - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_images_100k\100k\test (test images)
  - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_labels (detection labels)
  - c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\bdd100k_dr

## 5. Build the Model Architecture from the Paper

The main reproduction path uses the official YOLOP model definition from `hustvl/YOLOP`. This notebook also keeps the local `yolop_model.py` as a fallback sanity-check implementation.

In [6]:
# Add the cloned repository to sys.path for official YOLOP imports.
if YOLOP_REPO.exists() and str(YOLOP_REPO) not in sys.path:
    sys.path.insert(0, str(YOLOP_REPO))

# Patch the official config to point to the local BDD100K root.
def patch_default_config(config_path: Path, data_root: Path):
    if not config_path.exists():
        print("Config not found:", config_path)
        return
    text = config_path.read_text(encoding="utf-8")
    replacements = {
        "_C.DATA_DIR = 'data'": f"_C.DATA_DIR = r'{data_root}'",
        "_C.DATASET.ROOT = 'data'": f"_C.DATASET.ROOT = r'{data_root}'",
        "_C.DATASET.ROOT_DIR = 'data'": f"_C.DATASET.ROOT_DIR = r'{data_root}'",
    }
    for old, new in replacements.items():
        if old in text:
            text = text.replace(old, new)
    config_path.write_text(text, encoding="utf-8")
    print("Patched config:", config_path)

patch_default_config(YOLOP_REPO / "lib" / "config" / "default.py", DATA_ROOT)

# Try to import the official model entry point.
try:
    from lib.config import cfg
    from lib.models import get_net
    official_model = get_net(cfg)
    print("Official YOLOP model instantiated.")
    print(type(official_model))
except Exception as exc:
    official_model = None
    print("Official YOLOP import failed:", exc)

# Local fallback model for quick structural sanity checks.
try:
    from yolop_model import get_net as get_local_net
    local_model = get_local_net()
    print("Local fallback model instantiated.")
    print(type(local_model))
except Exception as exc:
    local_model = None
    print("Local fallback import failed:", exc)


Patched config: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\YOLOP\lib\config\default.py
Official YOLOP import failed: No module named 'torch'
Local fallback import failed: No module named 'torch'


## 6. Train with the Paper’s Hyperparameters

The paper trains the official YOLOP model on BDD100K using the repo defaults and the paper-format annotation folders.

In [7]:
runs_dir = ROOT / "runs"
runs_dir.mkdir(exist_ok=True)

train_cmd = [sys.executable, "tools/train.py"]
print("Training command:", " ".join(train_cmd))
print("Run it from:", YOLOP_REPO)
print("Expected outputs will land under the repo's runs/ or configured log directory.")

# Uncomment the next line when the paper-format annotations and dependencies are ready.
# run(train_cmd, cwd=YOLOP_REPO)


Training command: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\venv\Scripts\python.exe tools/train.py
Run it from: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\YOLOP
Expected outputs will land under the repo's runs/ or configured log directory.


## 7. Evaluate Using the Paper’s Test Procedure

The paper evaluates the trained model on the BDD100K validation/test split using the official YOLOP test and demo scripts, then reports detection and segmentation metrics.

In [8]:
test_cmd = [sys.executable, "tools/test.py", "--weights", "weights/End-to-end.pth"]
demo_cmd = [sys.executable, "tools/demo.py", "--source", str(DATA_ROOT / "images" / "val")]

print("Test command:", " ".join(test_cmd))
print("Demo command:", " ".join(demo_cmd))
print("When you have a trained checkpoint, run the test command from the repo root.")

# Uncomment the next line when a trained checkpoint exists.
# run(test_cmd, cwd=YOLOP_REPO)

# Uncomment to reproduce the paper-style visual demo on validation images.
# run(demo_cmd, cwd=YOLOP_REPO)


Test command: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\venv\Scripts\python.exe tools/test.py --weights weights/End-to-end.pth
Demo command: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\venv\Scripts\python.exe tools/demo.py --source c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\datasets\images\val
When you have a trained checkpoint, run the test command from the repo root.


## 8. Compare Reproduced Metrics with Reported Results

This section compares the reproduced metrics with the paper's reported values for detection and both segmentation tasks.

In [9]:
reported_metrics = {
    "det_mAP50": 0.765,
    "da_iou": 0.915,
    "ll_iou": 0.705,
}

# Fill this from the output of tools/test.py or from the validation log after a real run.
reproduced_metrics = {
    "det_mAP50": None,
    "da_iou": None,
    "ll_iou": None,
}

for key, reported in reported_metrics.items():
    actual = reproduced_metrics.get(key)
    if actual is None:
        print(f"{key}: reproduced value not set yet; paper value = {reported}")
    else:
        delta = actual - reported
        print(f"{key}: reproduced={actual:.4f}, paper={reported:.4f}, delta={delta:+.4f}")

# You can also load a metrics JSON if you save one from the evaluation cell.
metrics_file = ROOT / "reproduced_metrics.json"
if metrics_file.exists():
    loaded = json.loads(metrics_file.read_text())
    print("Loaded metrics JSON:", loaded)


det_mAP50: reproduced value not set yet; paper value = 0.765
da_iou: reproduced value not set yet; paper value = 0.915
ll_iou: reproduced value not set yet; paper value = 0.705


## 9. Save Trained Model, Predictions, and Logs

Persist the checkpoint, predictions, metrics, and logs so you can compare repeated runs and keep the reproduction traceable.

In [10]:
artifacts_dir = ROOT / "artifacts"
artifacts_dir.mkdir(exist_ok=True)

print("Suggested artifact paths:")
print(" - model checkpoint:", artifacts_dir / "yolop_best.pt")
print(" - metrics JSON:", artifacts_dir / "reproduced_metrics.json")
print(" - predictions:", artifacts_dir / "predictions/")
print(" - logs:", artifacts_dir / "logs/")

# Final verification: print model output shapes for one sample.
try:
    import torch
    sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    model_to_check = official_model if official_model is not None else local_model
    if model_to_check is None:
        print("No model available for verification.")
    else:
        model_to_check.eval()
        with torch.no_grad():
            result = model_to_check(sample)
        print("Verification output type:", type(result))
        if isinstance(result, tuple) and len(result) == 3:
            det_out, da_seg, ll_seg = result
            print("Detection output type:", type(det_out))
            if isinstance(det_out, tuple):
                print("Detection predictions shape:", det_out[0].shape)
            print("Drivable-area segmentation shape:", da_seg.shape)
            print("Lane-line segmentation shape:", ll_seg.shape)
except Exception as exc:
    print("Verification skipped:", exc)


Suggested artifact paths:
 - model checkpoint: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\artifacts\yolop_best.pt
 - metrics JSON: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\artifacts\reproduced_metrics.json
 - predictions: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\artifacts\predictions
 - logs: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\artifacts\logs
Verification skipped: No module named 'torch'


## 10. Comprehensive Test & Evaluation (Paper Comparison)

This section runs the full evaluation pipeline following the paper's methodology and compares your reproduced results with the official YOLOP paper metrics.


In [11]:
import json
import subprocess
from pathlib import Path
from datetime import datetime

# Paper's official reported metrics (from YOLOP paper)
PAPER_METRICS = {
    "detection_mAP50": 0.765,      # mAP@0.5 on BDD100K val
    "detection_mAP75": 0.548,      # mAP@0.75 on BDD100K val
    "da_segmentation_IoU": 0.915,  # Drivable area segmentation IoU
    "ll_segmentation_IoU": 0.705,  # Lane line segmentation IoU
}

print("=" * 80)
print("YOLOP PAPER REPRODUCTION - EVALUATION PROTOCOL")
print("=" * 80)
print("\nPaper's Official Metrics (BDD100K validation set):")
for metric, value in PAPER_METRICS.items():
    print(f"  {metric:.<35} {value:.4f}")
print("\n" + "=" * 80)


YOLOP PAPER REPRODUCTION - EVALUATION PROTOCOL

Paper's Official Metrics (BDD100K validation set):
  detection_mAP50.................... 0.7650
  detection_mAP75.................... 0.5480
  da_segmentation_IoU................ 0.9150
  ll_segmentation_IoU................ 0.7050



In [12]:
def run_model_test(weights_path, data_root, img_size=640, device="cuda"):
    """
    Run YOLOP model test following the official test.py procedure.
    
    Args:
        weights_path: Path to model checkpoint (.pt or .pth)
        data_root: Root directory of BDD100K dataset
        img_size: Input image size
        device: 'cuda' or 'cpu'
    
    Returns:
        dict with detection and segmentation metrics
    """
    if not Path(weights_path).exists():
        print(f"ERROR: Weights not found at {weights_path}")
        return None
    
    print(f"\n[TEST] Running model test with weights: {weights_path}")
    print(f"[TEST] Dataset root: {data_root}")
    print(f"[TEST] Image size: {img_size}")
    print(f"[TEST] Device: {device}")
    
    # Add YOLOP repo to path for imports
    if str(YOLOP_REPO) not in sys.path:
        sys.path.insert(0, str(YOLOP_REPO))
    
    try:
        import torch
        from lib.models import get_net
        from lib.config import cfg
        
        # Configure model
        cfg.MODEL.BACKBONE = "darknet"
        cfg.TEST.IMG_SIZE = img_size
        
        # Load model
        model = get_net(cfg)
        checkpoint = torch.load(weights_path, map_location=device)
        
        # Handle different checkpoint formats
        if isinstance(checkpoint, dict) and 'model' in checkpoint:
            model.load_state_dict(checkpoint['model'])
        else:
            model.load_state_dict(checkpoint)
        
        model = model.to(device)
        model.eval()
        
        print("[TEST] Model loaded successfully")
        return model
        
    except Exception as e:
        print(f"[ERROR] Failed to load model: {e}")
        import traceback
        traceback.print_exc()
        return None


# Check for available weight files
print("\n[WEIGHTS CHECK] Scanning for available model checkpoints...")
weight_sources = [
    YOLOP_REPO / "weights" / "End-to-end.pth",
    ROOT / "artifacts" / "yolop_best.pt",
    ROOT / "runs" / "exp" / "weights" / "best.pt",
]

available_weights = []
for wp in weight_sources:
    if wp.exists():
        print(f"  ✓ Found: {wp}")
        available_weights.append(wp)
    else:
        print(f"  ✗ Not found: {wp}")

if available_weights:
    selected_weights = available_weights[0]
    print(f"\n[WEIGHTS] Using: {selected_weights}")
else:
    print("\n[ERROR] No model weights found! Download or train first.")
    selected_weights = None



[WEIGHTS CHECK] Scanning for available model checkpoints...
  ✓ Found: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\YOLOP\weights\End-to-end.pth
  ✗ Not found: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\artifacts\yolop_best.pt
  ✗ Not found: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\runs\exp\weights\best.pt

[WEIGHTS] Using: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\YOLOP\weights\End-to-end.pth


In [13]:
def compute_metrics_from_official_test():
    """
    Run YOLOP's official test.py to compute detection and segmentation metrics.
    This is the gold-standard way to evaluate per the paper.
    """
    print("\n" + "=" * 80)
    print("STEP 1: Running Official YOLOP test.py")
    print("=" * 80)
    
    if not YOLOP_REPO.exists():
        print("[ERROR] YOLOP repo not found at", YOLOP_REPO)
        return None
    
    test_script = YOLOP_REPO / "tools" / "test.py"
    if not test_script.exists():
        print("[ERROR] test.py not found at", test_script)
        return None
    
    if selected_weights is None:
        print("[ERROR] No model weights available")
        return None
    
    # Run the official test command
    cmd = [
        sys.executable,
        str(test_script),
        "--weights", str(selected_weights),
        "--img-size", str(IMG_SIZE),
        "--device", DEVICE,
    ]
    
    print(f"\n[COMMAND] {' '.join(cmd)}")
    print(f"[CWD] {YOLOP_REPO}\n")
    
    try:
        result = subprocess.run(
            cmd,
            cwd=YOLOP_REPO,
            capture_output=True,
            text=True,
            timeout=3600  # 1 hour timeout
        )
        
        print(result.stdout)
        if result.stderr:
            print("[STDERR]", result.stderr)
        
        if result.returncode == 0:
            print("\n✓ Test completed successfully")
        else:
            print(f"\n✗ Test failed with return code {result.returncode}")
        
        return result
        
    except subprocess.TimeoutExpired:
        print("[ERROR] Test timed out after 1 hour")
        return None
    except Exception as e:
        print(f"[ERROR] Failed to run test: {e}")
        return None


def extract_metrics_from_output(output_text):
    """
    Parse metrics from test.py output.
    Looks for lines containing detection and segmentation metrics.
    """
    metrics = {}
    lines = output_text.split('\n') if output_text else []
    
    # Pattern matching for common metric formats
    import re
    
    for line in lines:
        # Detection metrics: mAP@0.5, mAP@0.75, etc.
        if 'mAP' in line.upper() or 'map' in line.lower():
            print(f"  [DETECTION] {line.strip()}")
            
        # Segmentation IoU metrics
        if 'iou' in line.lower() or 'intersection' in line.lower():
            print(f"  [SEGMENTATION] {line.strip()}")
            
        # Drivable area specific
        if 'drivable' in line.lower() or 'da_' in line.lower():
            print(f"  [DRIVABLE AREA] {line.strip()}")
            
        # Lane line specific
        if 'lane' in line.lower() or 'll_' in line.lower():
            print(f"  [LANE LINE] {line.strip()}")
    
    return metrics


print("\n✓ Evaluation functions defined. Ready to test model.")



✓ Evaluation functions defined. Ready to test model.


### Execute Full Evaluation Pipeline

**IMPORTANT**: Before running, ensure:
1. ✓ BDD100K dataset is properly structured in `datasets/` with the paper-format folders
2. ✓ Model weights are available (End-to-end.pth, yolop_best.pt, or trained checkpoint)
3. ✓ CUDA is available (or DEVICE will fall back to CPU)

Run the cell below to execute the test and generate metrics.


In [14]:
# ============================================================================
# STEP 1: Verify Dataset Structure
# ============================================================================
print("\n" + "=" * 80)
print("STEP 1: Dataset Structure Verification")
print("=" * 80)

required_dataset_folders = {
    "bdd100k_images_100k/100k/train": "Training images",
    "bdd100k_images_100k/100k/val": "Validation images",
    "bdd100k_images_100k/100k/test": "Test images",
    "bdd100k_labels/100k/train": "Detection labels (train)",
    "bdd100k_labels/100k/val": "Detection labels (val)",
    "bdd100k_labels/100k/test": "Detection labels (test)",
    "bdd100k_drivable_maps/labels/train": "Drivable area annotations (train)",
    "bdd100k_drivable_maps/labels/val": "Drivable area annotations (val)",
    "bdd100k_drivable_maps/color_labels/train": "Drivable area color labels (train)",
    "bdd100k_drivable_maps/color_labels/val": "Drivable area color labels (val)",
}

dataset_ready = True
for folder, desc in required_dataset_folders.items():
    path = DATA_ROOT / folder
    if path.exists():
        count = len(list(path.glob("*")))
        print(f"  ✓ {folder:.<40} ({count} items)")
    else:
        print(f"  ✗ {folder:.<40} MISSING")
        dataset_ready = False

if not dataset_ready:
    print("\n⚠️  WARNING: Some dataset folders are missing!")
    print("   The official paper uses the full BDD100K with prepared annotations.")
    print("   Download from: https://bdd-data.berkeley.edu/")

# ============================================================================
# STEP 2: Run Official Test
# ============================================================================
print("\n" + "=" * 80)
print("STEP 2: Running Official YOLOP Test")
print("=" * 80)

if selected_weights is not None:
    test_result = compute_metrics_from_official_test()
    if test_result and test_result.stdout:
        metrics_output = test_result.stdout
        print("\n[OUTPUT] Extracting metrics from test output...")
        extract_metrics_from_output(metrics_output)
else:
    print("⚠️  Skipping test - no model weights available")
    print("   Get weights from: https://github.com/hustvl/YOLOP/releases")
    print("   Or train your own using: python YOLOP/tools/train.py")



STEP 1: Dataset Structure Verification
  ✗ bdd100k_images_100k/100k/train.......... MISSING
  ✗ bdd100k_images_100k/100k/val............ MISSING
  ✗ bdd100k_images_100k/100k/test........... MISSING
  ✗ bdd100k_labels/100k/train............... MISSING
  ✗ bdd100k_labels/100k/val................. MISSING
  ✗ bdd100k_labels/100k/test................ MISSING
  ✗ bdd100k_drivable_maps/labels/train...... MISSING
  ✗ bdd100k_drivable_maps/labels/val........ MISSING
  ✗ bdd100k_drivable_maps/color_labels/train MISSING
  ✗ bdd100k_drivable_maps/color_labels/val.. MISSING

⚠️  WARNING: Some dataset folders are missing!
   The official paper uses the full BDD100K with prepared annotations.
   Download from: https://bdd-data.berkeley.edu/

STEP 2: Running Official YOLOP Test

STEP 1: Running Official YOLOP test.py

[COMMAND] c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\venv\Scripts\python.exe c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artific

### Step 3: Compare Results with Paper

After running `test.py`, manually extract the metrics from the output above and fill them in below.


In [15]:
# ============================================================================
# STEP 3: Manual Metric Entry & Comparison
# ============================================================================
print("\n" + "=" * 80)
print("STEP 3: Results Comparison with Paper")
print("=" * 80)

# TODO: After running test.py, extract these values from the output and fill them in
reproduced_metrics = {
    "detection_mAP50": None,       # Extract from test output: mAP@0.5
    "detection_mAP75": None,       # Extract from test output: mAP@0.75
    "da_segmentation_IoU": None,   # Drivable area IoU
    "ll_segmentation_IoU": None,   # Lane line IoU
}

print("\nPaper's Official Results:")
print("-" * 80)
print(f"{'Metric':<30} {'Paper Value':<15} {'Your Result':<15} {'Δ (diff)':<15}")
print("-" * 80)

all_reported = True
for metric, paper_value in PAPER_METRICS.items():
    your_value = reproduced_metrics[metric]
    
    if your_value is None:
        print(f"{metric:<30} {paper_value:<15.4f} {'[pending]':<15} {'—':<15}")
        all_reported = False
    else:
        delta = your_value - paper_value
        delta_pct = (delta / paper_value * 100) if paper_value != 0 else 0
        print(f"{metric:<30} {paper_value:<15.4f} {your_value:<15.4f} {delta:+.4f} ({delta_pct:+.1f}%)")

print("-" * 80)

if not all_reported:
    print("\n📝 HOW TO FILL IN YOUR RESULTS:")
    print("   1. Run test.py (see cell above)")
    print("   2. Look for output lines containing:")
    print("      - 'mAP@0.5' or 'mAP50' → detection_mAP50")
    print("      - 'mAP@0.75' or 'mAP75' → detection_mAP75")
    print("      - 'IoU' with 'drivable' or 'da_' → da_segmentation_IoU")
    print("      - 'IoU' with 'lane' or 'll_' → ll_segmentation_IoU")
    print("   3. Update the reproduced_metrics dict above")
    print("   4. Re-run this cell to see the comparison")
else:
    print("\n✓ All metrics reported!")
    avg_delta = sum(
        (reproduced_metrics[m] - PAPER_METRICS[m]) / PAPER_METRICS[m]
        for m in PAPER_METRICS.keys()
        if reproduced_metrics[m] is not None
    ) / len(PAPER_METRICS)
    
    print(f"\nAverage deviation from paper: {avg_delta * 100:+.2f}%")
    
    if abs(avg_delta) < 0.05:
        print("✓✓✓ Excellent! Reproduction closely matches paper results.")
    elif abs(avg_delta) < 0.10:
        print("✓✓ Good match. Minor variations are expected.")
    else:
        print("⚠️  Significant deviation from paper. Check:")
        print("   - Dataset preprocessing")
        print("   - Model weights initialization")
        print("   - Training hyperparameters")



STEP 3: Results Comparison with Paper

Paper's Official Results:
--------------------------------------------------------------------------------
Metric                         Paper Value     Your Result     Δ (diff)       
--------------------------------------------------------------------------------
detection_mAP50                0.7650          [pending]       —              
detection_mAP75                0.5480          [pending]       —              
da_segmentation_IoU            0.9150          [pending]       —              
ll_segmentation_IoU            0.7050          [pending]       —              
--------------------------------------------------------------------------------

📝 HOW TO FILL IN YOUR RESULTS:
   1. Run test.py (see cell above)
   2. Look for output lines containing:
      - 'mAP@0.5' or 'mAP50' → detection_mAP50
      - 'mAP@0.75' or 'mAP75' → detection_mAP75
      - 'IoU' with 'drivable' or 'da_' → da_segmentation_IoU
      - 'IoU' with 'lane' or 'll_

In [16]:
# ============================================================================
# STEP 4: Save Evaluation Report
# ============================================================================
print("\n" + "=" * 80)
print("STEP 4: Saving Evaluation Report")
print("=" * 80)

def save_evaluation_report():
    """Save detailed evaluation report to JSON and Markdown."""
    
    report = {
        "timestamp": datetime.now().isoformat(),
        "paper_metrics": PAPER_METRICS,
        "reproduced_metrics": reproduced_metrics,
        "configuration": {
            "img_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "device": DEVICE,
            "seed": SEED,
        },
        "dataset": {
            "root": str(DATA_ROOT),
            "splits_available": dataset_ready,
        },
        "model": {
            "weights": str(selected_weights) if selected_weights else None,
            "type": "YOLOP",
            "repo": str(YOLOP_REPO),
        }
    }
    
    # Calculate deltas
    report["comparison"] = {}
    for metric in PAPER_METRICS.keys():
        paper_val = PAPER_METRICS[metric]
        your_val = reproduced_metrics.get(metric)
        
        if your_val is not None:
            delta = your_val - paper_val
            delta_pct = (delta / paper_val * 100) if paper_val != 0 else 0
            report["comparison"][metric] = {
                "paper": paper_val,
                "reproduced": your_val,
                "delta": delta,
                "delta_percent": delta_pct,
            }
    
    # Save JSON report
    report_json = artifacts_dir / "evaluation_report.json"
    report_json.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"✓ JSON report saved: {report_json}")
    
    # Save Markdown report
    report_md = artifacts_dir / "evaluation_report.md"
    md_content = f"""# YOLOP Paper Reproduction Evaluation Report

**Generated**: {report['timestamp']}

## Configuration
- Image Size: {report['configuration']['img_size']}
- Batch Size: {report['configuration']['batch_size']}
- Epochs: {report['configuration']['epochs']}
- Device: {report['configuration']['device']}
- Random Seed: {report['configuration']['seed']}

## Model & Dataset
- Model: {report['model']['type']}
- Weights: {report['model']['weights']}
- Dataset Root: {report['dataset']['root']}
- Dataset Ready: {report['dataset']['splits_available']}

## Metrics Comparison

| Metric | Paper Value | Your Result | Δ | Δ (%) |
|--------|-------------|-------------|---|-------|
"""
    
    for metric, paper_val in PAPER_METRICS.items():
        your_val = reproduced_metrics.get(metric)
        if your_val is not None:
            delta = your_val - paper_val
            delta_pct = (delta / paper_val * 100) if paper_val != 0 else 0
            md_content += f"| {metric} | {paper_val:.4f} | {your_val:.4f} | {delta:+.4f} | {delta_pct:+.2f}% |\n"
        else:
            md_content += f"| {metric} | {paper_val:.4f} | [pending] | — | — |\n"
    
    report_md.write_text(md_content, encoding="utf-8")
    print(f"✓ Markdown report saved: {report_md}")
    
    return report


# Uncomment to save the report
report = save_evaluation_report()
print("\nTo save the evaluation report, uncomment the line above and re-run this cell.")



STEP 4: Saving Evaluation Report
✓ JSON report saved: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\artifacts\evaluation_report.json
✓ Markdown report saved: c:\Users\joaco\OneDrive\Documents\facultad\Vision\TP-final-Vision-Artificial\artifacts\evaluation_report.md

To save the evaluation report, uncomment the line above and re-run this cell.


## Summary: How to Run the Full Evaluation

This section provides a complete guide to test and evaluate your YOLOP reproduction against the official paper metrics.

### Prerequisites

1. **Dataset**: Your BDD100K structure
   ```
   datasets/
   ├── bdd100k_images_100k/100k/
   │   ├── train/
   │   ├── val/
   │   └── test/
   ├── bdd100k_labels/100k/
   │   ├── train/
   │   ├── val/
   │   └── test/
   └── bdd100k_drivable_maps/
       ├── labels/train, labels/val
       └── color_labels/train, color_labels/val
   ```

2. **Model Weights**: One of:
   - `YOLOP/weights/End-to-end.pth` (official pretrained)
   - `artifacts/yolop_best.pt` (your trained checkpoint)
   - `runs/exp/weights/best.pt` (training output)

3. **Environment**: 
   ```bash
   cd "/media/vanafa/1TB/Workspace/Cuatri 7/Visión/TP Final"
   source .venv/bin/activate
   ```

### Running the Evaluation

**Step 1**: Open this notebook and run all cells up to "Step 2: Running Official YOLOP Test"

**Step 2**: The notebook will:
- ✓ Verify dataset structure (checks your actual folders)
- ✓ Locate available model weights
- ✓ Run `YOLOP/tools/test.py` (this takes 30-60 mins depending on dataset size)

**Step 3**: After test completes, extract metrics from the output:
- Look for lines with `mAP@0.5`, `mAP@0.75`, `IoU`
- Copy these values to the `reproduced_metrics` dict in "Step 3" cell
- Re-run that cell to see the comparison table

**Step 4**: Save the report by uncommenting the line in "Step 4" cell

### Expected Output

✓ **Evaluation Report** showing:
```
┌─────────────────────────────────────────────────────┐
│ Metric              │ Paper    │ Your Result │ Δ (%) │
├─────────────────────────────────────────────────────┤
│ detection_mAP50     │ 0.7650   │ ?           │ ?     │
│ detection_mAP75     │ 0.5480   │ ?           │ ?     │
│ da_segmentation_IoU │ 0.9150   │ ?           │ ?     │
│ ll_segmentation_IoU │ 0.7050   │ ?           │ ?     │
└─────────────────────────────────────────────────────┘
```

✓ **Artifact Files**:
- `artifacts/evaluation_report.json` - Machine-readable results
- `artifacts/evaluation_report.md` - Human-readable markdown

### Troubleshooting

**"Model weights not found"**
- Download from: https://github.com/hustvl/YOLOP/releases/download/v1.0/End-to-end.pth
- Place in `YOLOP/weights/End-to-end.pth`

**"Dataset folders missing"**
- Your dataset folders are at:
  - `datasets/bdd100k_images_100k/100k/`
  - `datasets/bdd100k_labels/100k/`
  - `datasets/bdd100k_drivable_maps/`

**"Test script fails"**
- Ensure CUDA is available: `python -c "import torch; print(torch.cuda.is_available())"`
- Or use CPU: Set `DEVICE = "cpu"` in cell 1

**"Out of memory"**
- Reduce BATCH_SIZE or IMG_SIZE
- Reduce GPU memory usage with `DEVICE="cpu"`